# Brand Purchase Probability Model

## Expected Relevant Variables

Based on the problem type, the most relevant variables are likely:

1. **`rfm_total` and individual scores**: High-value customers (high RFM) tend to buy premium brands
2. **`brand_code`**: Customer's favorite brand
3. **`brand_variety` and `wallet_hhi`**: Indicates if the customer is monogamous or explorer
4. **RFM Segment**: "Champions" and "Loyal" likely buy the brand more
5. **`top_brand_share`**: If their main brand matches Marca 0
6. **`frequency` and `monetary`**: Frequent and high-spending customers

### Model Setup

Selected brand: **Marca 0**

`SELECTED_BRAND`

In [0]:
CATALOG = "workspace"           
SCHEMA  = "default"              
TABLE   = "customer_profiles"
PURCHASE_TABLE = "prueba_bavaria_compras"

# Selected brand for purchase probability modeling
SELECTED_BRAND = "Marca 0" 

TEST_SIZE    = 0.25
RANDOM_STATE = 42

FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
FULL_PURCHASE_TABLE = f"{CATALOG}.{SCHEMA}.{PURCHASE_TABLE}"
print(f"Customer table: {FULL_TABLE}")
print(f"Purchase table: {FULL_PURCHASE_TABLE}")
print(f"Target brand: {SELECTED_BRAND}")

### **Target Variable Creation**
**Target = 1**: Customer has purchased the selected brand at least once (55%)

**Target = 0**: Customer has never purchased the brand (45%)
- Distribution: 55% buyers, 45% non-buyers (balanced)

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, trim, upper, to_date, max as spark_max, countDistinct

# Load customer profiles (use same as EDA)
sdf_customers = spark.table(FULL_TABLE)
print(f"Customer profiles: {sdf_customers.count():,} customers, {len(sdf_customers.columns)} columns")

# Load and aggregate purchase data efficiently
selected_brand_clean = SELECTED_BRAND.strip().upper()

# Single pass through purchase data: get last date and brand info per customer
purchase_summary = (spark.table(FULL_PURCHASE_TABLE)
                   .select("CUSTOMER", "ORDER_DATE", "BRAND")
                   .withColumn("brand_clean", trim(upper(col("BRAND"))))
                   .withColumn("order_date", to_date(col("ORDER_DATE")))
                   .groupBy("CUSTOMER")
                   .agg(
                       spark_max("order_date").alias("last_purchase_date"),
                       spark_max((col("brand_clean") == selected_brand_clean).cast("int")).alias("bought_brand")
                   ))

print(f"Customers who bought '{SELECTED_BRAND}': {purchase_summary.filter(col('bought_brand') == 1).count():,}")

# Restrict to customers present in EDA profiles
sdf_with_target = (sdf_customers
                   .join(purchase_summary, 
                         sdf_customers.customer == purchase_summary.CUSTOMER,
                         how="left")
                   .withColumn("target", col("bought_brand").cast("int"))
                   .fillna({"target": 0})
                   .drop("CUSTOMER", "bought_brand"))

# Convert to pandas
df = sdf_with_target.toPandas()

print(f"\nFinal dataset: {len(df):,} customers")
print(f"Target distribution:")
print(df['target'].value_counts().sort_index())
print(f"Purchase rate: {df['target'].mean():.2%}")
print(f"Date range: {df['last_purchase_date'].min()} to {df['last_purchase_date'].max()}")
df.head()

### **Predictor Variables (Features)**

In [0]:
# Columns to exclude from features (identifiers, dates, and target)
DROP_COLS = ['customer', 'rfm_code', 'top_brand', 'rfm_segment', 'brand_affinity', 'target', 'last_purchase_date']

# Simple encoding for top brand (customer's most purchased brand)
df['brand_code'] = df['top_brand'].astype('category').cat.codes

# One-hot encoding for rfm_segment (Champions, Loyal, etc.)
df = pd.concat([df, pd.get_dummies(df['rfm_segment'], prefix='seg')], axis=1)

# One-hot encoding for brand_affinity (Monogamous, Explorer, Mixed)
df = pd.concat([df, pd.get_dummies(df['brand_affinity'], prefix='affinity')], axis=1)

# Create feature matrix X (only numeric columns)
X = (df.drop(columns=[c for c in DROP_COLS if c in df.columns])
       .select_dtypes(include=[np.number, 'bool']))

# Create target variable y
y = df['target'].values

print(f"X shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"\ny (target) shape: {y.shape}")
print(f"Customers who bought {SELECTED_BRAND}: {y.sum():,} ({y.mean():.2%})")
print(f"Customers who did NOT buy {SELECTED_BRAND}: {(1-y).sum():,} ({(1-y.mean()):.2%})")

### 4. **Train/Test Split**
- 75% train (3,159 customers) / 25% test (1,054 customers)
- **Stratified split**: Maintains the same proportion of buyers in train and test

In [0]:
# Temporal train/test split: last 2 weeks as test set
max_date = df['last_purchase_date'].max()
cutoff_date = max_date - pd.Timedelta(days=14)

print(f"Max purchase date: {max_date}")
print(f"Cutoff date (2 weeks back): {cutoff_date}")

# Train: customers whose last purchase was BEFORE the last 2 weeks
# Test: customers whose last purchase was IN the last 2 weeks
train_mask = df['last_purchase_date'] < cutoff_date
test_mask = df['last_purchase_date'] >= cutoff_date

X_tr = X[train_mask]
X_te = X[test_mask]
y_tr = y[train_mask]
y_te = y[test_mask]

print(f"\nTrain set: {len(X_tr):,} customers (purchases before {cutoff_date})")
print(f"Test set:  {len(X_te):,} customers (purchases in last 2 weeks)")
print(f"\nBuyers in train: {y_tr.sum():,} ({y_tr.mean():.2%})")
print(f"Buyers in test:  {y_te.sum():,} ({y_te.mean():.2%})")

In [0]:
%pip install xgboost

### **Model**

In [0]:
import xgboost as xgb

pos_weight = (1 - y_tr.mean()) / y_tr.mean()

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    tree_method='hist',
    random_state=RANDOM_STATE,
    scale_pos_weight=pos_weight,
)

model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print("Training completed.")

### **Evaluation**
- AUC-ROC, Precision, Recall, F1-Score
- Confusion matrix
- Feature importance

In [0]:
X_tr

In [0]:
corr_with_target = df[[*X.columns, 'target']].corr()['target'].drop('target')
display(corr_with_target)

In [0]:
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve, confusion_matrix, classification_report,
)

proba = model.predict_proba(X_te)[:, 1]
auc   = roc_auc_score(y_te, proba)

# Threshold 0.50
pred_05 = (proba >= 0.5).astype(int)
p_05 = precision_score(y_te, pred_05)
r_05 = recall_score(y_te, pred_05)
f_05 = f1_score(y_te, pred_05)

# Optimal threshold by F1
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_te, proba)
f1s = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-9)
best_idx = int(np.nanargmax(f1s[:-1]))
t_opt = float(thr_curve[best_idx])

pred_opt = (proba >= t_opt).astype(int)
p_opt = precision_score(y_te, pred_opt)
r_opt = recall_score(y_te, pred_opt)
f_opt = f1_score(y_te, pred_opt)

metrics = pd.DataFrame({
    'metric':    ['AUC-ROC',
                  'Precision @ 0.50', 'Recall @ 0.50', 'F1 @ 0.50',
                  f'Precision @ opt({t_opt:.3f})',
                  f'Recall @ opt({t_opt:.3f})',
                  'Optimal F1'],
    'value':     [auc, p_05, r_05, f_05, p_opt, r_opt, f_opt],
})
display(metrics)

### **Confusion Matrix and Classification Report**

In [0]:
print("Confusion matrix @ 0.50")
print(confusion_matrix(y_te, pred_05))
print()
print("Classification report @ 0.50")
print(classification_report(y_te, pred_05, digits=4))
print()
print(f"Confusion matrix @ optimal threshold ({t_opt:.3f})")
print(confusion_matrix(y_te, pred_opt))

### **Top Important Features**

In [0]:
importances = (pd.Series(model.feature_importances_, index=X.columns)
                 .sort_values(ascending=False)
                 .head(20)
                 .rename('importance')
                 .to_frame())
importances['variable'] = importances.index
display(importances[['variable', 'importance']])